# Micelles Under the Microscope

### Setting Things Up

In [ ]:
# @title
!python -V
!pip install -q openmm MDAnalysis py3Dmol ipywidgets

# install compiler
!apt-get -qq update
!apt-get -qq install -y gfortran

# remove existing directory if it exists
import os
import shutil
if os.path.exists("packmol-master"):
    print("Removing existing packmol-master directory...")
    shutil.rmtree("packmol-master")

# download latest Packmol source
!wget -q https://github.com/m3g/packmol/archive/refs/heads/master.zip
!unzip -q master.zip

# compile
%cd packmol-master
!make
%cd ..

# add to PATH

os.environ["PATH"] = "/content/packmol-master:" + os.environ["PATH"]
!which packmol

from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display

#SIMULATION TEMPERATURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=298.0,  # default value
    description='Simulation Temperature (K):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTemperature
    simTemperature = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTemperature = temp_text.value*kelvin

#BOXSIZE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=48.0,  # default value
    description='Box Size (Angstrom):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global boxsize
    boxsize = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
boxsize = temp_text.value

#TIMESTEP
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=2.0,  # default value
    description='Time step (fs):',
    step=0.5,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTimestep
    simTimestep = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTimestep = temp_text.value*femtoseconds

#PRESSURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=1.0,  # default value
    description='Simulation pressure (atm):',
    step=0.10,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simPressure
    simPressure = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simPressure = temp_text.value*atmospheres

#STEPS
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=100000,  # default value
    description='# MD steps:',
    step=100,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simNumSteps
    simNumSteps = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simNumSteps = temp_text.value

In [ ]:
density=0.99705
water=int(boxsize**3*density*1e6*(1e-10)**3/(18.01)*6.02214e23)
print("Box size of",boxsize,"angstroms at density,",density,"g/cm3 requires ",water," water molecules")

In [ ]:
print(f"Simulation size: {water} waters")
print(f"Box size: {boxsize} Ang.")
print(f"Simulation temperature: {simTemperature}")
print(f"Simulation pressure: {simPressure}")
print(f"Simulation time step: {simTimestep}")
print(f"# MD steps: {simNumSteps} ; Total simulation length: {simNumSteps * simTimestep} = {simNumSteps*simTimestep / 1000 / 1000 / femtoseconds * nanoseconds}.")
pdbFile = 'system.pdb'

#### Building the Initial Micelle Structure

In [ ]:
with open("packmol.in", "w") as f:
    f.write("tolerance 2.0\n")
    f.write("filetype pdb\n")
    f.write("output system.pdb\n")
    f.write("\n")

    # Define box and centre
    L = boxsize + 30
    cx = cy = cz = L / 2.0

    # Radii for micelle bias
    r_outer = 0.35 * L     # whole SDS region
    r_head_inner = 0.25 * L
    r_core = 0.15 * L

    # SDS molecules
    f.write("structure sds.pdb\n")
    f.write("  number 60\n")

    # Keep whole molecule in spherical region
    f.write("  inside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_outer))

    # Tail-end carbon (atom 39) pulled toward centre
    f.write("  atoms 39\n")
    f.write("    inside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_core))
    f.write("  end atoms\n")

    # Sulfur (atom 1) pushed outward
    f.write("  atoms 1\n")
    f.write("    outside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_head_inner))
    #f.write("    inside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_outer))
    f.write("  end atoms\n")

    f.write("end structure\n")

    # Sodium ions (no constraint)
    f.write("structure na+.pdb\n")
    f.write("  number 60\n")
    f.write("  inside cube 2. 2. 2. {0}\n".format(L))
    f.write("end structure\n")

    # Water
    f.write("structure water.pdb\n")
    f.write("  number {0}\n".format(int(water)))
    f.write("  inside cube 2. 2. 2. {0}\n".format(L))
    f.write("end structure\n")

    f.write("\n")
    f.write("pbc {0} {0} {0}\n".format(L))

In [ ]:
#%%bash
!packmol < packmol.in > packmol.out

In [ ]:
with open('packmol.out', 'r') as f:
    text = f.read()
    print(text)

In [ ]:
import py3Dmol

with open("system.pdb", "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=900, height=600)
# crucial change: keepH=True
view.addModel(pdb_data, "pdb", {"keepH": True})

# Default style: water as sticks, ions as spheres
view.setStyle({}, {})
view.setStyle({"elem": "O"}, {"stick": {"radius": 0.05}})
view.addStyle({"elem": "H"}, {"stick": {"radius": 0.05}})
view.setStyle({"resn": "SDS"}, {"stick": {"radius": 0.15}})
view.addStyle({"elem": "Na"}, {"sphere": {"scale": 0.7, "color": "blue"}})

view.zoomTo()
view.show()

#### Defining the MD Simulation

In [ ]:
pdb = PDBFile('system.pdb')

In [ ]:
forcefield = ForceField('charmm36.xml','charmm36/water.xml')

In [ ]:
system = forcefield.createSystem(pdb.topology,nonbondedMethod=PME,nonbondedCutoff=14*angstrom,constraints=HBonds,rigidWater=True)

In [ ]:
system.addForce(MonteCarloBarostat(simPressure, simTemperature, 1))

In [ ]:
integrator = LangevinMiddleIntegrator(simTemperature, 1/picosecond, simTimestep)

In [ ]:
simulation = Simulation(pdb.topology, system, integrator)

In [ ]:
simulation.context.setPositions(pdb.positions)

#### Energy Minimisation

In [ ]:
print("Running Energy Minimisation:")
t0 = time.time()
simulation.minimizeEnergy(maxIterations=0)
t1 = time.time()
minTime = t1-t0
print("Minimisation took {} seconds".format(minTime))

#### Running the MD Simulation

In [ ]:
simulation.reporters.append(DCDReporter('sds.dcd', 10))
simulation.reporters.append(StateDataReporter(
    'sds_data.csv',
    10,
    step=True,
    time=True,
    temperature=True,
    potentialEnergy=True,
    kineticEnergy=True,
    totalEnergy=True,
    volume=True,
    density=True
))

In [ ]:
state = simulation.context.getState(getPositions=False, getVelocities=False, getForces=False, getEnergy=False, getParameters=False)
box_vectors = state.getPeriodicBoxVectors()
print(f'Periodic box vectors: {box_vectors}')

In [ ]:
# Run the simulation:
print('Start Simulation')
t0 = time.time()
simulation.step(simNumSteps)
t1 = time.time()
simTime = t1-t0
print("Simulation took {} seconds for {} timesteps".format(simTime,simNumSteps))

### Analysis

#### Watching the Micelle Form

In [ ]:
import MDAnalysis as mda
import py3Dmol

u = mda.Universe("system.pdb", "sds.dcd")

stride = 10

# Keep only SDS + Na+, omit water
sel = u.select_atoms("resname SDS or name NA or name Na or element Na")

# Build XYZ trajectory in memory
xyz_frames = []

for ts in u.trajectory[::stride]:
    lines = [f"{len(sel)}", f"Frame {ts.frame}"]

    for atom in sel:
        x, y, z = atom.position
        name = atom.name.upper()

        if name.startswith("H"):
            elem = "H"
        elif name.startswith("O"):
            elem = "O"
        elif name.startswith("NA"):
            elem = "Na"
        elif name.startswith("CL"):
            elem = "Cl"
        elif name.startswith("S"):
            elem = "S"
        elif name.startswith("C"):
            elem = "C"
        else:
            elem = atom.name[0]

        lines.append(f"{elem} {x:.3f} {y:.3f} {z:.3f}")

    xyz_frames.append("\n".join(lines))

# Combine all frames into one string
xyz_data = "\n".join(xyz_frames)



#### Has the System Equilibrated?

In [ ]:
# Visualise
view = py3Dmol.view(width=900, height=550)
view.addModelsAsFrames(xyz_data, "xyz")

# Global style for SDS
view.setStyle({}, {"stick": {"radius": 0.12}})

# Make sulfur stand out a bit
view.addStyle({"elem": "S"}, {"sphere": {"scale": 0.35}})

# Highlight Na+
view.addStyle({"elem": "Na"}, {"sphere": {"scale": 0.7, "color": "blue"}})

view.zoomTo()
view.animate({"interval": 100})
view.show()

In [ ]:


# Load the data from sds_data.csv
df = pd.read_csv('sds_data.csv')

# Assuming 'Time (ps)' is the time column
time_col = 'Time (ps)'
if 'Time (ps)' not in df.columns:
    # Try 'Time' or other common names if 'Time (ps)' is not found
    if 'Time' in df.columns:
        time_col = 'Time'
    else:
        print("Warning: 'Time (ps)' column not found. Please check CSV header.")
        # Fallback to using step if no time column is clearly identifiable
        time_col = 'Step'

x_data = df[time_col]

# Identify columns to plot (excluding 'Step', '# step', and the time column itself)
columns_to_exclude = ['#"Step"', time_col]
columns_to_plot = [col for col in df.columns if col not in columns_to_exclude]

# Create subplots dynamically with 2 columns
num_plots = len(columns_to_plot)
num_cols_per_row = 2
num_rows = math.ceil(num_plots / num_cols_per_row)

fig, axes = plt.subplots(num_rows, num_cols_per_row, figsize=(15, num_rows * 4), sharex=True)

# Ensure axes is a flattened iterable array even if there's only one subplot or one row
if num_plots == 0:
    print("No columns to plot other than time and step.")
    plt.close(fig)
else:
    # Flatten the axes array for easier iteration if it's 2D
    if num_rows > 1 or num_cols_per_row > 1:
        axes = axes.flatten()
    else:
        axes = [axes] # Make it a list if it's a single subplot

    for i, col_name in enumerate(columns_to_plot):
        axes[i].plot(x_data, df[col_name])
        axes[i].set_ylabel(col_name)
        axes[i].grid(True)

    # Hide any unused subplots
    for j in range(num_plots, len(axes)):
        fig.delaxes(axes[j])

    # Set common x-label for the bottom subplots that are visible
    for i in range((num_rows - 1) * num_cols_per_row, num_plots):
        if i < len(axes):
            axes[i].set_xlabel(f'{time_col} (ps)' if time_col == 'Time' else time_col)

    fig.suptitle('MD Simulation Data Over Time', y=1.02, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.98]) # Adjust layout to prevent title overlap
    plt.show()

#### Micelle Size

In [ ]:
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt

u = mda.Universe("system.pdb", "sds.dcd")

# select micelle (SDS only)
micelle = u.select_atoms("resname SDS")

rg = []
time = []

for ts in u.trajectory:
    rg.append(micelle.radius_of_gyration())
    time.append(ts.time)  # ps (if available)

rg = np.array(rg)

plt.plot(time, rg)
plt.xlabel("Time (ps)")
plt.ylabel("Radius of gyration (Å)")
plt.title("Micelle size over time")
plt.show()

#### Micelle Shape

#### Micelle Stability

#### Counterion Behaviour

#### Dynamics of the Water-Micelle Interface

#### Exercise - Micelle Formation in Aqueous Electrolytes ####

<span style="color:red;">Using the sample code above copy, paste and edit a new code to:

1. Simulate the formation of an SDS micelle in ?? M NaCl. 
1. Calculate the size, shape, stability etc., as you did above in pure water. 
1. Create a single plot ...</span>


##### Important Notes #####
1. Insert as many code cells in the jupyter notebook below as you need. Try and keep your code as 'clean' as possible - it will help you solve and debug problems if and when they happen!
1. If you get stuck, ask your demonstrator for help!
1. <span style="color:red;">REALLY IMPORTANT - make sure that you change all "nacl" to "naf" and/or "nai" in your new code below for NaF and NaI. Otherwise you will destroy the NaCl coordination numbers you just calculated!</span>

## Discussion Questions and Information for Preparing Your Report ##

In your report, consider and discuss the following questions based on the data you have created above:

1. 

_You may find the following references useful for preparing your report:_

1. 